# Diagnostic : encoder les codifications `PAY_n` en catégories

**Question** : depuis la v6, `PAY_1` à `PAY_6` sont traités comme des nombres (-2 à 8), ce qui suppose une même échelle de risque de -2 à 8. Or l'ordre entre -2, -1 et 0 n'est pas documenté, et 1 est un statut provisoire de la banque. Une régression logistique ne peut leur donner qu'**une seule pente** ; le KNN, le SVM et le MLP raisonnent en distances. Les arbres, eux, coupent déjà à des seuils et catégorisent naturellement ces valeurs.

**Variantes comparées**, sur les colonnes du scénario retenu `ml_4` :

| | Encodage des `PAY_n` | Colonnes ajoutées |
|---|---|---|
| A | nombres (v6, actuel) | — |
| F | **trois catégories par mois** : « à jour » (-2, -1 et 0 réunis), « 1 », « 2 et plus » | — |
| G | les mêmes trois catégories | `TYPE_USAGE` et `PAY_habituel`, qui reprennent la distinction comptant / crédit perdue dans « à jour » |

La pratique courante ramène les clients à jour à 0 ; la première itération l'a testée (`S4`) : gain pour la régression logistique, légère perte pour les arbres, alors privés de la distinction comptant / crédit qu'aucune autre colonne ne portait. La variante G la redonne.

**Méthode** : réglages figés à ceux de `ml_4`, mêmes 5 plis, score décisionnel F2 au taux de rappel minimal comparé pli par pli à la variante A. Modèles : régression logistique, KNN et MLP (les plus concernés), CatBoost et RandomForest (témoins). Le SVM, trop lent, n'est testé que si le résultat est prometteur. Rien n'est écrit dans le fichier de résultats, le test n'est pas lu.

## Données, variables et pipeline de `ml_4`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")
print(f"Clients du train : {len(train)} ; colonnes disponibles : {train.shape[1]}")

In [ ]:
# ==============================================================================
# VARIABLES DU SCÉNARIO : seule cellule à modifier pour changer les colonnes
# ==============================================================================

# Colonnes ajoutées par ce scénario, rangées selon leur encodage
ajout_num = ["ratio_PAY_BILL_global"]   # nombres, montants, ratios, compteurs
ajout_log = []   # montants positifs ou nuls très étirés : logarithme, puis mise à l'échelle
ajout_log_signe = []   # montants qui peuvent être négatifs (avoirs) : logarithme signé, puis mise à l'échelle
ajout_cat = ["FLAG_OUVERTURE", "FLAG_DEGEL", "MOIS_ACTIVATION"]   # catégories sans ordre (une colonne par catégorie), indicateurs 0 / 1
ajout_ord = []   # catégories dans un ordre naturel (encodage ordinal)

# Colonnes du socle retirées par ce scénario
retirees = []

# ------------------------------------------------------------------------------
# Socle : les 23 variables d'origine (ne pas modifier)
# PAY_n : des nombres déjà ordonnés et régulièrement espacés (-2 à 8, un cran par mois de retard), mis à l'échelle comme les autres (v6)
# PAY_AMTn : paiements très étirés (médiane d'environ 2 000 NT$, maximum de plus de 400 000 NT$), passés au logarithme (v7)
# BILL_AMTn : factures, traitées comme les paiements qu'elles appellent, avec un logarithme signé (avoirs négatifs) (v8)
socle_num = ["LIMIT_BAL", "AGE"] + [f"PAY_{n}" for n in range(1, 7)]
socle_log = [f"PAY_AMT{n}" for n in range(1, 7)]
socle_log_signe = [f"BILL_AMT{n}" for n in range(1, 7)]
socle_cat = ["SEX", "EDUCATION", "MARRIAGE"]
socle_ord = []

# ------------------------------------------------------------------------------
# Remplacements décidés dans docs/colonnes_creees.md (colonne « Adaptation et usage ML ») : {colonne: (valeurs remplacées, valeur)}
# Faits dans le pipeline, avant la transformation propre au type de la colonne (mise à l'échelle, logarithme, catégories…),
# et donc enregistrés avec le modèle. Valeur précise décidée par le métier, jamais calculée sur les clients (ni médiane ni moyenne).
# Actifs seulement si la colonne est une variable du scénario, quelle que soit sa liste. Un vide s'écrit np.nan ; une catégorie
# s'écrit en texte. À compléter à chaque nouvelle décision
remplacements = {
    "ratio_PAY_BILL_global": ([np.nan], 100),          # dû nul ou négatif : rien à payer, client à jour
    "ratio_PAY_BILL_median": ([-1], 100),              # seule la facture de septembre est due
    "ratio_PAY_BILL_regularite": ([np.nan, -1], 100),  # une seule facture due, ou « Première facture »
}

num_col = [c for c in socle_num if c not in retirees] + ajout_num
log_col = [c for c in socle_log if c not in retirees] + ajout_log
log_signe_col = [c for c in socle_log_signe if c not in retirees] + ajout_log_signe
cat_col = [c for c in socle_cat if c not in retirees] + ajout_cat
ord_col = [c for c in socle_ord if c not in retirees] + ajout_ord
variables = num_col + log_col + log_signe_col + cat_col + ord_col

# Contrôles : colonnes présentes dans le train, aucune en double, ni identifiant ni cible
absentes = [c for c in variables + retirees if c not in train.columns]
doublons = sorted({c for c in variables if variables.count(c) > 1})
interdites = [c for c in variables if c in ("ID", "dpnm")]
assert not absentes, f"Colonnes absentes du train : {absentes} (à créer dans creation_datasets_ML.ipynb)"
assert not doublons, f"Colonnes en double : {doublons}"
assert not interdites, f"Colonnes interdites comme variables : {interdites}"
# Logarithme : montants positifs ou nuls, une fois les valeurs remplacées mises de côté
for c in log_col:
    valeurs = train[c].dropna()
    if c in remplacements:
        valeurs = valeurs[~valeurs.isin([v for v in remplacements[c][0] if not pd.isna(v)])]
    assert (valeurs >= 0).all(), f"Le logarithme ne s'applique qu'à des montants positifs ou nuls : {c}"
# Remplacements : aucune variable ne garde un vide sans remplacement décidé
remplacees = [c for c in variables if c in remplacements]
vides_sans_remplacement = [c for c in variables if train[c].isna().any()
                           and not (c in remplacements and any(pd.isna(v) for v in remplacements[c][0]))]
assert not vides_sans_remplacement, f"Colonnes avec des vides, sans remplacement décidé dans docs/colonnes_creees.md : {vides_sans_remplacement}"

X_train = train[variables].reset_index(drop=True)
y_train = train["dpnm"].reset_index(drop=True)

print(f"Variables        : {len(variables)} ({len(num_col)} numériques, {len(log_col)} au logarithme, {len(log_signe_col)} au logarithme signé, {len(cat_col)} catégorielles, {len(ord_col)} ordinales)")
print(f"Ajoutées         : {ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord if ajout_num + ajout_log + ajout_log_signe + ajout_cat + ajout_ord else 'aucune'}")
print(f"Retirées         : {retirees if retirees else 'aucune'}")
print(f"Remplacements    : {', '.join(f'{c} ({remplacements[c][0]} -> {remplacements[c][1]})' for c in remplacees) if remplacees else 'aucun'}")
print(f"Défauts          : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

In [ ]:
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
import numpy as np
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
    "ord": lambda: make_pipeline(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), StandardScaler()),
}
COLONNES_PAR_TYPE = {"num": num_col, "log": log_col, "log_signe": log_signe_col, "nom": cat_col, "ord": ord_col}

def avec_remplacement(colonne, transformation):
    # Valeurs remplacées par la valeur décidée (cellule « variables du scénario »), puis transformation propre au type
    valeurs, valeur = remplacements[colonne]
    # Le vide d'abord : SimpleImputer refuse une colonne qui contient des vides quand il cherche une autre valeur
    valeurs = sorted(valeurs, key=lambda v: not pd.isna(v))
    return make_pipeline(*[SimpleImputer(missing_values=v, strategy="constant", fill_value=valeur) for v in valeurs], transformation)

# Une branche par type pour les colonnes sans remplacement, une branche par colonne avec remplacement.
# Sans remplacement (le socle), le pipeline est exactement celui de la v8
preprocessor = ColumnTransformer(transformers=
    [(nom, TRANSFORMATIONS[nom](), [c for c in colonnes if c not in remplacements]) for nom, colonnes in COLONNES_PAR_TYPE.items()]
    + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[nom]()), [c])
       for nom, colonnes in COLONNES_PAR_TYPE.items() for c in colonnes if c in remplacements])

# Objectif de détection : part des défauts que le modèle doit trouver (60 % pour commencer, objectif 70 %)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

def score_decision(estimator, X, y):
    # Précision au seuil qui détecte au moins RAPPEL_MINIMAL des défauts (score optimisé par la boucle)
    proba = estimator.predict_proba(X)[:, 1]
    signales = proba >= seuil_pour_rappel(y, proba)
    return np.asarray(y)[signales].mean()

scorers = {
    "decision": score_decision,
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles lancés dans ce notebook : retirer un nom pour ne pas le lancer (le reste du notebook s'adapte)
MODELES_A_TESTER = ["LogisticRegression", "SVM", "MLPClassifier", "KNN", "RandomForest", "CatBoost"]

def modeles_de_base():
    # Les 6 modèles d'origine, avec leurs réglages de la première itération ; seuls ceux de MODELES_A_TESTER sont rendus
    tous = {
        "LogisticRegression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "SVM": make_pipeline(preprocessor, CalibratedClassifierCV(estimator=SVC(class_weight="balanced", random_state=42), ensemble=False)),
        "MLPClassifier": make_pipeline(preprocessor, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "KNN": make_pipeline(preprocessor, KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        "RandomForest": make_pipeline(preprocessor, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(preprocessor, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

In [ ]:
from IPython.display import Markdown, display
from sklearn.metrics import fbeta_score

y = y_train.to_numpy()
plis = list(cv.split(X_train, y_train))   # les mêmes 5 plis que tous les scénarios
PAY = [f"PAY_{n}" for n in range(1, 7)]

def f2_au_rappel_minimal(y_vrai, proba):
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

def categories_pay(valeurs):
    # Trois catégories par mois : à jour (-2, -1, 0), 1 (statut provisoire), 2 et plus (en retard)
    v = np.asarray(valeurs, dtype=float)
    return np.where(v >= 2, "2 et plus", np.where(v == 1, "1", "à jour"))

def preprocesseur_variante(variante):
    if variante == "A":
        types = COLONNES_PAR_TYPE
        branches_pay = []
    else:
        ajout = ["TYPE_USAGE", "PAY_habituel"] if variante == "G" else []
        types = dict(COLONNES_PAR_TYPE, num=[c for c in num_col if c not in PAY], nom=cat_col + ajout)
        branches_pay = [("pay_categories", make_pipeline(FunctionTransformer(categories_pay, feature_names_out="one-to-one"),
                                                          OneHotEncoder(sparse_output=False, handle_unknown="ignore")), PAY)]
    return ColumnTransformer(transformers=
        [(nom, TRANSFORMATIONS[nom](), [c for c in colonnes if c not in remplacements]) for nom, colonnes in types.items()]
        + [(f"remplacement_{c}", avec_remplacement(c, TRANSFORMATIONS[nom]()), [c]) for nom, colonnes in types.items() for c in colonnes if c in remplacements]
        + branches_pay)

COLONNES_VARIANTE = {"A": variables, "F": variables, "G": variables + ["TYPE_USAGE", "PAY_habituel"]}

MODELES_FIGES = {   # réglages retenus dans ml_4
    "LogisticRegression": lambda: LogisticRegression(C=0.01, max_iter=1000, class_weight="balanced", random_state=42),
    "KNN": lambda: KNeighborsClassifier(n_neighbors=384, weights="uniform", n_jobs=-1),
    "MLPClassifier": lambda: MLPClassifier(hidden_layer_sizes=(32,), alpha=1.0, max_iter=500, random_state=42, early_stopping=True),
    "CatBoost": lambda: CatBoostClassifier(depth=3, iterations=445, l2_leaf_reg=33.8, learning_rate=0.0133, scale_pos_weight=scale_pos,
                                           random_state=42, verbose=0, thread_count=-1, allow_writing_files=False),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=150, class_weight="balanced", random_state=42, n_jobs=-1),
}

f2 = {}
for variante, colonnes in COLONNES_VARIANTE.items():
    for nom, fabrique in MODELES_FIGES.items():
        f2[(variante, nom)] = []
        for entrainement, validation in plis:
            pipeline = make_pipeline(preprocesseur_variante(variante), fabrique())
            pipeline.fit(train.iloc[entrainement][colonnes], y[entrainement])
            f2[(variante, nom)].append(f2_au_rappel_minimal(y[validation], pipeline.predict_proba(train.iloc[validation][colonnes])[:, 1]))
    print(f"Variante {variante} : " + " ; ".join(f"{nom} {np.mean(f2[(variante, nom)]):.3f}" for nom in MODELES_FIGES))

## Comparaison pli par pli avec l'encodage actuel (A)

Pour chaque modèle, puis en meilleur contre meilleur : gain moyen du score décisionnel F2 sur la variante A, écart-type des gains et plis gagnés. Verdict comme dans les scénarios.

In [ ]:
def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

def meilleur(variante):
    return max(MODELES_FIGES, key=lambda nom: np.mean(f2[(variante, nom)]))

lignes = []
for variante in ("F", "G"):
    for nom in list(MODELES_FIGES) + ["Meilleur contre meilleur"]:
        ref, comp = (meilleur("A"), meilleur(variante)) if nom == "Meilleur contre meilleur" else (nom, nom)
        gains = np.array(f2[(variante, comp)]) - np.array(f2[("A", ref)])
        lignes.append({"Variante": variante, "Modèle": nom if nom != "Meilleur contre meilleur" else f"{nom} ({ref} → {comp})",
                       "Score A": round(np.mean(f2[("A", ref)]), 3), "Score variante": round(np.mean(f2[(variante, comp)]), 3),
                       "Gain": f"{gains.mean():+.3f} ± {gains.std():.3f}", "Plis gagnés": f"{(gains > 0).sum()} sur 5", "Verdict": verdict(gains)})
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

**Comment lire** : une ligne par modèle et par variante. Si la régression logistique, le KNN ou le MLP gagnent avec F ou G, et que les arbres ne perdent pas, l'encodage en catégories est à retenir pour la suite (v9, appliquée à la version finale et à sa référence). La comparaison G contre F dit si `TYPE_USAGE` et `PAY_habituel` deviennent utiles une fois la distinction comptant / crédit retirée des `PAY_n`.